In [ ]:
import glob
import json
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from collections import defaultdict

sns.set_theme(style="whitegrid")

In [ ]:
with open("../../data/accession_translator.json", "r") as s:
    accession_translator = json.load(s)
with open("../../data/tool_translator.json", "r") as s:
    datasets = json.load(s)

In [ ]:
def parse_stats_file(stats_path):
    results = defaultdict(list)
    with open(stats_path, 'r') as stream:
        for line in stream:
            if line.startswith("#= Summary for dataset:"):
                datapath = line.split()[-1]
            if "level" in line:
                level, stats = line.split("level:")
                results["datapath"].append(datapath)
                results["level"].append(level.strip())
                results["sensitivity"].append(float(stats.split()[0]))
                results["precision"].append(float(stats.split()[2]))
    return pd.DataFrame.from_dict(results)

In [ ]:
stats_paths = glob.glob("../../intermediates/gffcompare_benchmarking/*/*.stats")
dfs = []
for path in stats_paths:
    accession = path.split('/')[-2]
    stats_df = parse_stats_file(path)
    stats_df["dataset"] = stats_df["datapath"].str.split('/').str[-2]
    stats_df["accession"] = accession
    stats_df["species"] = accession_translator[accession]
    dfs.append(stats_df)
all_stats = pd.concat(dfs)
all_stats["species"] = pd.Categorical(all_stats["species"], accession_translator.values())
all_stats = all_stats.sort_values("dataset")
all_stats["F1 score"] = 2*all_stats["precision"]*all_stats["sensitivity"]/(all_stats["precision"]+all_stats["sensitivity"])

In [ ]:
all_stats["dataset"] = pd.Categorical(all_stats["dataset"], categories=datasets.keys(), ordered=True)
all_stats["dataset"] = all_stats["dataset"].map(datasets)

In [ ]:
levels = {
    "Base": "Nucleotide",
    "Exon": "Exon",
    "Locus": "Gene",
}

In [ ]:
selected_stats = all_stats[all_stats["level"].isin(["Base", "Exon", "Locus"])]
selected_stats["level"] = pd.Categorical(selected_stats["level"], categories=levels.keys(), ordered=True)
selected_stats["level"] = selected_stats["level"].map(levels)
selected_stats.drop(columns=["datapath","accession"]).to_csv("../../tables/Table_S5.tsv", sep='\t',index=False)

In [ ]:
summary = selected_stats.groupby(["level","dataset"])[["sensitivity","precision","F1 score"]].agg(['mean','std'])

collapsed = summary.copy()
collapsed.columns = ['_'.join(col) for col in summary.columns]  # flatten columns

for metric in ["sensitivity","precision","F1 score"]:
    collapsed[metric] = (
        collapsed[f'{metric}_mean'].round(1).astype(str)
        + " ± " +
        collapsed[f'{metric}_std'].round(1).astype(str)
    )

collapsed = collapsed[["sensitivity","precision","F1 score"]]
collapsed.to_csv("../../tables/Table_2.tsv", sep='\t')

In [ ]:
tool_palette = {
    "ANNEVO": "#009E73",
    "AUGUSTUS": "#0072B2",
    "BRAKER3": "#FF81C6",
    "Helixer": "#462288",
    "geneML": "#FF822F",
}

In [ ]:
data = selected_stats.sort_values("dataset").rename(columns={"dataset":"Gene prediction tool","level":"Level"})
plot = sns.relplot(data=data, x="precision", y="sensitivity", col="species", col_wrap=3, height=3, hue="Gene prediction tool", style="Level", s=150, alpha=0.8, palette=tool_palette)
plot.set(xlim=(0, 100), ylim=(0, 100))
plot.set_ylabels("recall")
plot.set_titles("{col_name}", size=11)
plot._legend.set_title(None)
plt.savefig("../../figures/Figure_2.svg")

In [ ]:
def get_annotation_counts(tsv_file):
    counts = defaultdict(int)
    annotations = set()
    last_gene_id = None
    with open(tsv_file, 'r') as s:
        next(s)
        for line in s:
            fields = line.strip().split("\t")
            gene_id = fields[0]
            annotation = fields[3]
            if gene_id == last_gene_id:
                annotations.add(annotation)
            else:
                if last_gene_id is not None:
                    if "Complete" in annotations:
                        counts["Complete"] += 1
                    elif "Partial" in annotations:
                        counts["Partial"] += 1
                    else:
                        counts["None"] += 1
                annotations = {annotation}
            last_gene_id = gene_id

    if last_gene_id is not None:
        if "Complete" in annotations:
            counts["Complete"] += 1
        elif "Partial" in annotations:
            counts["Partial"] += 1
        else:
            counts["None"] += 1

    return counts

In [ ]:
paths = glob.glob("../../intermediates/benchmarking_results/*/pfam/*annotations.tsv")
records = []

for path in paths:
    counts = get_annotation_counts(path)
    genome = path.split("/")[-1].replace("_annotations.tsv", "")
    dataset = path.split("/")[-3]

    for annotation_type in ["partial", "complete", "none"]:
        records.append({
            "type": annotation_type,
            "count": counts.get(annotation_type.capitalize(), 0),
            "genome": accession_translator[genome],
            "dataset": dataset,
        })

annotation_df = pd.DataFrame(records)

In [ ]:
datasets["reference"] = "Reference"
order = ["reference"] + [key for key in datasets.keys() if key != "reference"]
annotation_df["dataset"] = pd.Categorical(annotation_df["dataset"], categories=order, ordered=True)
annotation_df["dataset"] = annotation_df["dataset"].map(datasets)

type_labels = {
    "none": "None",
    "partial": "Partial",
    "complete": "Complete",
}

label_order = ["None", "Partial", "Complete"]
annotation_df["type"] = annotation_df["type"].map(type_labels)

In [ ]:
table_df = annotation_df.copy()
table_df["type"] = pd.Categorical(table_df["type"], categories=label_order, ordered=True)

summary_table = (
    table_df
    .pivot_table(index=["genome", "type"], columns="dataset", values="count", aggfunc="sum", fill_value=0)
    .reset_index()
    .sort_values(["genome", "type"])
)

dataset_columns = [datasets[key] for key in order if datasets[key] in summary_table.columns]
summary_table = summary_table[["genome", "type"] + dataset_columns]

summary_table.to_csv("../../tables/Table_S6.tsv", sep='\t', index=False)

In [ ]:
tool_palette2 = {
    "ANNEVO": "#5CB08F",
    "AUGUSTUS": "#338EC1",
    "BRAKER3": "#FF9AD1",
    "Helixer": "#6B4EA0",
    "geneML": "#FF9B59",
    "Reference": "#B4B4B4",
}

In [ ]:
g = sns.catplot(data=annotation_df, y="type", x="count", hue="dataset", col="genome", kind="bar",
            palette=tool_palette2, saturation=1, sharex=False, order=label_order)

g.set_titles("{col_name}")
g.set_axis_labels("Number of genes", "PFAM domain annotation")

g.savefig("../../figures/Figure_S2.svg")